# BERTopic — Vietnamese Fintech News

**Flow:** load data → raw-text embeddings → UMAP → HDBSCAN → CountVectorizer → c-TF-IDF → BERTopic → 12-case Grid Search → evaluate → visualize → export.

- Select exactly one field: `title`, `snippet`, or `raw_article`. Rerun from the data cell when switching fields.
- No stopword removal or segmentation **before embeddings**.
- Seed (`42`)
- Each model's settings appear right next to its constructor; inspect one cell at a time.

## 0. Install pinned dependencies (once)
Enable a GPU runtime in Colab if available. After installing, restart the kernel if prompted.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

x = torch.tensor([1, 2, 3], device="cuda")
print("GPU test:", x * 2)

CUDA available: True
GPU: Tesla T4
GPU test: tensor([2, 4, 6], device='cuda:0')


In [2]:
%pip install -q --timeout 60 --retries 2 bertopic "sentence-transformers==3.3.1" "transformers==4.46.3" umap-learn hdbscan underthesea gensim plotly pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Imports & reproducibility

In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import json
import hashlib
import random
import platform
import unicodedata
from pathlib import Path
from itertools import product
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
import transformers
import sentence_transformers

from IPython.display import display
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from underthesea import word_tokenize

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Kaggle dataset configuration
DATASET_NAME = "Fintech8kk"
TEXT_FIELD = "raw_article"  # "title", "snippet", "raw_article"
DATE_COLUMN = "pub_date"

# Locate uploaded dataset automatically
csv_files = sorted(Path("/kaggle/input").rglob("fintech_news_final.csv"))

if not csv_files:
    raise FileNotFoundError(
        "Dataset not found. Check that your CSV is attached under Kaggle Input."
    )

DATA_PATH = csv_files[0]

# Output configuration
OUTPUT_DIR = Path("/kaggle/working/bertopic_fintechnews")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID_SEARCH = True
SHOW_PLOTS = True

# Load dataset
df = pd.read_csv(DATA_PATH)

assert TEXT_FIELD in df.columns, f"Missing column: {TEXT_FIELD}"
assert DATE_COLUMN in df.columns, f"Missing column: {DATE_COLUMN}"

# Environment information
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

print({
    name: version(name)
    for name in [
        "bertopic",
        "sentence-transformers",
        "umap-learn",
        "hdbscan",
        "gensim",
        "underthesea"
    ]
})

print("\nDataset:", DATASET_NAME)
print("Data path:", DATA_PATH)
print("Loaded rows:", f"{len(df):,}")
print("Text field:", TEXT_FIELD)
print("Output directory:", OUTPUT_DIR)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

display(df.head())

I0000 00:00:1795034549.684536     234 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Python: 3.13.16
PyTorch: 2.11.0+cu130
Transformers: 4.46.3
SentenceTransformers: 3.3.1
Device: cuda
{'bertopic': '0.17.4', 'sentence-transformers': '3.3.1', 'umap-learn': '0.5.12', 'hdbscan': '0.8.44', 'gensim': '4.4.0', 'underthesea': '9.5.0'}

Dataset: Fintech8kk
Data path: /kaggle/input/datasets/christiannam/fintech-news/fintech_news_final.csv
Loaded rows: 8,440
Text field: raw_article
Output directory: /kaggle/working/bertopic_fintechnews
CUDA available: True
GPU: Tesla T4


,url,title,snippet,raw_article,pub_date,curated_source_urls_mapped
0,f.vn/chu-the-mb-tu-tin-san-san-pham-apple-voi-...,Chủ thẻ MB tự tin săn sản phẩm Apple với ưu đã...,NaN,"Từ ngày 22/09 đến 31/10/2023, MB cho phép khác...",2023-09-23,[]
1,f.vn/cong-nghe-dang-thay-doi-thi-truong-ngan-h...,và VIB đang là ngân hàng tiên phong triển khai...,NaN,"Dù đã có những kết quả tích cực, ngành ngân hà...",2022-12-23,[]
2,f.vn/mo-tai-khoan-so-dep-vib-digi-dung-y-nhan-...,"hấp dẫn.""",NaN,Bạn đang mong muốn mở tài khoản ngân hàng nhan...,2022-02-24,[]
3,f.vn/ncb-tung-loat-uu-dai-tri-an-khach-hang-su...,NCB tung loạt ưu đãi tri ân khách hàng sử dụng...,NaN,Với tiêu chí Ngân hàng gọn gàng - Sống nhẹ nhà...,2023-04-19,[]
4,f.vn/nhieu-khach-hang-bi-tru-tien-tu-giao-dich...,"Nhiều khách hàng bị trừ tiền từ giao dịch lạ, ...",NaN,Vietcombank cho biết sẽ khoá thẻ của khách hàn...,2020-01-29,[]


## 3. Read data

In [4]:
df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")

# Test
# df = df.sample(n=200, random_state=42).copy()

required = {"url", TEXT_FIELD}
if missing := required - set(df.columns):
    raise KeyError(f"Missing columns: {missing}")

valid_rows = df["url"].str.strip().ne("") & df[TEXT_FIELD].str.strip().ne("")
df = df.loc[valid_rows].drop_duplicates(subset="url").set_index("url", drop=False)

docs = df[TEXT_FIELD].tolist()
doc_ids = df["url"].tolist()

print(f"{TEXT_FIELD}: {len(docs):,} documents")
print(f"Removed {len(valid_rows) - len(df):,} empty/duplicate rows")
display(df[[ TEXT_FIELD]].head(3))

raw_article: 8,440 documents
Removed 0 empty/duplicate rows


,raw_article
url,
f.vn/chu-the-mb-tu-tin-san-san-pham-apple-voi-uu-dai-tra-gop-0-dong-188230923120244284.c,"Từ ngày 22/09 đến 31/10/2023, MB cho phép khác..."
f.vn/cong-nghe-dang-thay-doi-thi-truong-ngan-hang-so-nhu-the-nao-20221223154954625.c,"Dù đã có những kết quả tích cực, ngành ngân hà..."
f.vn/mo-tai-khoan-so-dep-vib-digi-dung-y-nhan-ngay-500k-qua-tang-cuc-chat-20220223172757536.c,Bạn đang mong muốn mở tài khoản ngân hàng nhan...


## 4. Embedding model
Uses raw Vietnamese documents. The cache is reused only when the data and embedding configuration match.

In [5]:
import types
# 1. Embedding model: original text, no preprocessing
EMBEDDING_MODEL_ID = "dangvantuan/vietnamese-document-embedding"
MODEL_REVISION = "6fa4e2f8ed2d33120b0f4442cc81f8f973c3f56b"
EMBEDDING_BATCH_SIZE = 4
CHECKPOINT_EVERY = 500

EMBEDDING_NORMALIZE = True

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    device="cpu",
)
for module in embedding_model.modules():
    if hasattr(module, "position_ids") and isinstance(module.position_ids, torch.Tensor) and module.position_ids.ndim == 1:
        n = module.position_ids.numel()
        module.register_buffer(
            "position_ids",
            torch.arange(n, device=module.position_ids.device),
            persistent=False
        )

# Repair attention mask
def extended_attention_mask(self, mask, input_shape, device=None, dtype=None):
    dtype = dtype or next(self.parameters()).dtype
    return (1.0 - mask[:, None, None, :].to(dtype=dtype)) * torch.finfo(dtype).min

model = embedding_model[0].auto_model
layer = model.embeddings
config = model.config

# Reinitialize rotary embeddings
layer.rotary_emb = type(layer.rotary_emb)(
    dim=config.hidden_size // config.num_attention_heads,
    max_position_embeddings=config.max_position_embeddings,
    base=config.rope_theta,
    scaling_factor=config.rope_scaling["factor"],
    mixed_b=config.rope_scaling.get("mixed_b"),
)

model.get_extended_attention_mask = types.MethodType(
    extended_attention_mask, model
)

print("Max sequence length:", embedding_model.max_seq_length)

Max sequence length: 8192


In [6]:
# test = embedding_model.encode(
#     ["Đây là một bài viết về công nghệ tài chính."],
#     batch_size=1,
#     device="cpu"
# )

# print(test.shape)

# embedding_model = embedding_model.to("cuda")

# test = embedding_model.encode(
#     ["Đây là một bài viết về công nghệ tài chính."],
#     batch_size=1,
#     device="cuda"
# )

# print("GPU test:", test.shape)

In [7]:
# Embedding configuration
embedding_model.max_seq_length = 512
EMBEDDING_BATCH_SIZE = 8

embedding_file = OUTPUT_DIR / "embeddings.npy"
metadata_file = OUTPUT_DIR / "embeddings.json"

# Cache metadata
text_hash = hashlib.sha256("\0".join(docs).encode("utf-8")).hexdigest()

cache_info = {
    "text_hash": text_hash,
    "model": EMBEDDING_MODEL_ID,
    "max_seq_length": embedding_model.max_seq_length,
    "normalize": EMBEDDING_NORMALIZE,
}

# Load cached embeddings
cache_valid = False

if embedding_file.exists() and metadata_file.exists():
    try:
        saved_info = json.loads(metadata_file.read_text())
        if saved_info == cache_info:
            embeddings = np.load(embedding_file, allow_pickle=False)
            cache_valid = (
                embeddings.ndim == 2
                and len(embeddings) == len(docs)
                and np.isfinite(embeddings).all()
            )
    except (OSError, ValueError, EOFError, json.JSONDecodeError):
        cache_valid = False

# Generate embeddings
if not cache_valid:
    embeddings = embedding_model.encode(
        docs,
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=EMBEDDING_NORMALIZE,
        convert_to_numpy=True,
        device="cuda" if torch.cuda.is_available() else "cpu",
    ).astype(np.float32)

    np.save(embedding_file, embeddings)
    metadata_file.write_text(json.dumps(cache_info, indent=2))

print("Embeddings:", embeddings.shape, "| Cache hit:", cache_valid)

Batches:   0%|          | 0/1055 [00:00<?, ?it/s]

Embeddings: (8440, 768) | Cache hit: False


## 5. UMAP — dimensionality reduction

UMAP parameters are declared directly in the constructor below.

In [8]:
UMAP_NEIGHBORS = [15, 30, 50] if GRID_SEARCH else [15]

umap_model = UMAP(
    n_neighbors=15,     # baseline; search varies this parameter
    n_components=5,
    min_dist=0.0,       #The desired minimum distance between points in UMAP space
    metric="cosine",
    random_state=SEED,
    low_memory=True,
    n_jobs=1,
)
umap_model

UMAP(metric='cosine', min_dist=0.0, n_components=5, n_jobs=1, random_state=42)

## 6. HDBSCAN — component parameters

In [9]:
HDBSCAN_CLUSTER_SIZES = [10, 15, 30] if GRID_SEARCH else [15]

hdbscan_model = HDBSCAN(
    min_cluster_size=15,    # baseline; grid varies this
    min_samples=5,          # keep min_sample small to avoid cluster data into -1 group
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=False,
    core_dist_n_jobs=1,
)
hdbscan_model

HDBSCAN(core_dist_n_jobs=1, min_cluster_size=15, min_samples=5)

## 7. CountVectorizer — keywords only

Tokenize Vietnamese **only for topic representation and coherence**. The embedding model receives untouched text. No custom stopword list by default.

In [10]:
TOPIC_STOPWORDS = set()  # optional: only applies to keywords/coherence

def vietnamese_tokens(text):
  text = unicodedata.normalize("NFC", str(text))
  return [token for token in word_tokenize(text, format="text").lower().split()
            if len(token) > 1 and token not in TOPIC_STOPWORDS]

vectorizer_model = CountVectorizer(
    tokenizer=vietnamese_tokens,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 3),
    # min_df=2,
# Test
    min_df =1,

    max_features=30_000,
)
vectorizer_model

CountVectorizer(lowercase=False, max_features=30000, ngram_range=(1, 3),
                token_pattern=None,
                tokenizer=<function vietnamese_tokens at 0x7d5b2003c0e0>)

## 8. c-TF-IDF & BERTopic — assemble the components

In [11]:
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True,
)
topic_model

## 9. Evaluation — standard coherence metrics

Gensim coherence uses BERTopic's top words and the *same tokenization* on the reference texts. The topic `-1` (outliers) is excluded.

In [12]:
# 9. Evaluation: Coherence and Topic Diversity
import numpy as np
from itertools import combinations
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

TOP_WORDS = 20
CV_WINDOW = 110
IRBO_WEIGHT = 0.9

# 1. Reference corpus
reference_texts = [vietnamese_tokens(doc) for doc in docs]
dictionary = Dictionary(reference_texts)
reference_bow = [dictionary.doc2bow(tokens) for tokens in reference_texts]

if len(dictionary) == 0:
    raise ValueError("Evaluation dictionary is empty.")


# 2. Inverted Rank-Biased Overlap (IRBO)
def calculate_irbo(topic_words, p=IRBO_WEIGHT):
    """Mean pairwise inverted extrapolated RBO. Higher IRBO = greater topic diversity."""
    if not 0 < p < 1:
        raise ValueError("IRBO weight p must be between 0 and 1.")
    if len(topic_words) < 2:
        return np.nan

    scores = []
    for a, b in combinations(topic_words, 2):
        if len(a) != len(b):
            raise ValueError("IRBO requires equal-length topic lists.")

        depth = len(a)
        seen_a, seen_b = set(), set()
        weighted_overlap, final_agreement = 0.0, 0.0

        for d in range(1, depth + 1):
            seen_a.add(a[d - 1])
            seen_b.add(b[d - 1])
            agreement = len(seen_a & seen_b) / d
            weighted_overlap += agreement * (p ** (d - 1))
            final_agreement = agreement

        rbo = (1 - p) * weighted_overlap + final_agreement * (p ** depth)
        scores.append(1.0 - rbo)

    return float(np.mean(scores))


# 3. Evaluate BERTopic candidate
def evaluate(model, topics):
    # Extract top-20 terms, excluding outlier topic -1
    topic_words = []
    for topic_id in sorted(model.get_topics()):
        if topic_id == -1:
            continue
        terms = model.get_topic(topic_id)
        if terms:
            topic_words.append([word for word, _ in terms[:TOP_WORDS]])

    assignments = np.asarray(topics)
    n_topics = len(topic_words)

    # Clustering diagnostics
    out = {
        "n_topics": n_topics,
        "outlier_rate": float(np.mean(assignments == -1)),
        "coverage": float(np.mean(assignments != -1)),
        "c_v": np.nan,
        "u_mass": np.nan,
        "topic_diversity": np.nan,
        "irbo": np.nan,
    }

    if n_topics == 0:
        return out

    # Require the same top-N for all topics
    if any(len(t) != TOP_WORDS for t in topic_words):
        raise ValueError(f"Each topic must contain {TOP_WORDS} terms.")

    # 4. Topic Diversity
    unique_terms = {word for topic in topic_words for word in topic}
    out["topic_diversity"] = float(len(unique_terms) / (n_topics * TOP_WORDS))

    # 5. IRBO
    out["irbo"] = calculate_irbo(topic_words, p=IRBO_WEIGHT)

    # 6. Coherence vocabulary alignment
    coherence_words = [[word for word in topic if word in dictionary.token2id] for topic in topic_words]
    valid_coherence_topics = [topic for topic in coherence_words if len(topic) >= 2]

    out["coherence_topics"] = len(valid_coherence_topics)
    out["coherence_term_coverage"] = float(sum(map(len, coherence_words)) / (n_topics * TOP_WORDS))

    if len(valid_coherence_topics) == 0:
        return out

    # 7. C_V Coherence
    out["c_v"] = float(CoherenceModel(
        topics=valid_coherence_topics, texts=reference_texts, dictionary=dictionary,
        coherence="c_v", window_size=CV_WINDOW, topn=TOP_WORDS, processes=1
    ).get_coherence())

    # 8. UMass Coherence
    out["u_mass"] = float(CoherenceModel(
        topics=valid_coherence_topics, corpus=reference_bow, dictionary=dictionary,
        coherence="u_mass", topn=TOP_WORDS, processes=1
    ).get_coherence())

    return out


## 10. Train / Grid Search

For every combination, use the component configurations above and change only the searched parameters. One fixed seed (`42`). A CSV score is saved after each run.

In [13]:
# # SINGLE RUN — COARSE-GRAINED BERTopic
# GRID_SEARCH = False

# umap_model = UMAP(n_neighbors=30, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
# hdbscan_model = HDBSCAN(min_cluster_size=20, min_samples=10, metric="euclidean", cluster_selection_method="eom", prediction_data=True)

# topic_model = BERTopic(embedding_model=embedding_model, umap_model=umap_model, hdbscan_model=hdbscan_model,
#                       vectorizer_model=vectorizer_model, ctfidf_model=ctfidf_model, verbose=True)

# best_topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)

# best_config = {"n_neighbors": 30, "min_cluster_size": 20, "min_samples": 10}
# metrics = evaluate(topic_model, best_topics)

# print("Config:", best_config)
# print("Metrics:", metrics)
# print(topic_model.get_topic_info().head(20))

In [14]:
grid = list(product(UMAP_NEIGHBORS, HDBSCAN_CLUSTER_SIZES))
print(f"Training {len(grid)} configuration(s)")

scores = []
best_model = None
best_topics = None
best_config = None
best_rank = None

for i, (nn, cluster_size) in enumerate(grid, 1):
    print(f"\n[{i}/{len(grid)}] UMAP n_neighbors={nn} | HDBSCAN min_cluster_size={cluster_size}, min_samples = 5 ")
    # Preserve fixed parameters from the constructors above.
    candidate_umap = UMAP(**{**umap_model.get_params(), "n_neighbors": nn})
    candidate_hdbscan = HDBSCAN(**{**hdbscan_model.get_params(),
                                  "min_cluster_size": cluster_size})
    candidate = BERTopic(
        embedding_model=embedding_model,
        umap_model=candidate_umap,
        hdbscan_model=candidate_hdbscan,
        vectorizer_model=clone(vectorizer_model),
        ctfidf_model=clone(ctfidf_model),
        top_n_words=TOP_WORDS,
        calculate_probabilities=False,
        verbose=True,
    )
    labels, _ = candidate.fit_transform(docs, embeddings=embeddings)
    metrics = evaluate(candidate, labels)
    row = dict(n_neighbors=nn, min_cluster_size=cluster_size, **metrics)
    scores.append(row)
    pd.DataFrame(scores).to_csv(OUTPUT_DIR / "grid_search.csv", index=False)
    print(
    f"C_V={row['c_v']:.4f} | "
    f"UMass={row['u_mass']:.4f} | "
    f"TD={row['topic_diversity']:.4f} | "
    f"IRBO={row['irbo']:.4f} | "
    f"Topics={row['n_topics']} | "
    f"Outliers={row['outlier_rate']:.1%}"
)

    # Select by high C_v, then fewer outliers, then fewer topics.
    rank = (row['c_v'] if np.isfinite(row['c_v']) else -np.inf,
            -row['outlier_rate'], -row['n_topics'])
    if best_rank is None or rank > best_rank:
        best_model, best_topics, best_config, best_rank = candidate, np.array(labels), row, rank

topic_model = best_model
print("\nSelected:", best_config)
display(pd.DataFrame(scores).sort_values(["c_v", "outlier_rate", "n_topics"],
                                          ascending=[False, True, True]))

2026-11-18 20:56:34,477 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training 9 configuration(s)

[1/9] UMAP n_neighbors=15 | HDBSCAN min_cluster_size=10, min_samples = 5 


2026-11-18 20:57:10,388 - BERTopic - Dimensionality - Completed ✓
2026-11-18 20:57:10,390 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-18 20:57:10,566 - BERTopic - Cluster - Completed ✓
2026-11-18 20:57:10,573 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-18 21:04:55,350 - BERTopic - Representation - Completed ✓
2026-11-18 21:13:58,588 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


C_V=0.7088 | UMass=-2.5237 | TD=0.7566 | IRBO=0.9974 | Topics=198 | Outliers=25.5%

[2/9] UMAP n_neighbors=15 | HDBSCAN min_cluster_size=15, min_samples = 5 


2026-11-18 21:14:16,720 - BERTopic - Dimensionality - Completed ✓
2026-11-18 21:14:16,722 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-18 21:14:16,906 - BERTopic - Cluster - Completed ✓
2026-11-18 21:14:16,911 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-18 21:21:59,621 - BERTopic - Representation - Completed ✓
2026-11-18 21:31:09,927 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


C_V=0.7087 | UMass=-2.3433 | TD=0.7868 | IRBO=0.9968 | Topics=133 | Outliers=21.3%

[3/9] UMAP n_neighbors=15 | HDBSCAN min_cluster_size=30, min_samples = 5 


... (11 outputs left out)

C_V=0.7248 | UMass=-2.0432 | TD=0.8095 | IRBO=0.9963 | Topics=95 | Outliers=21.7%

[9/9] UMAP n_neighbors=50 | HDBSCAN min_cluster_size=30, min_samples = 5 


2026-11-18 23:13:27,148 - BERTopic - Dimensionality - Completed ✓
2026-11-18 23:13:27,150 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-18 23:13:27,343 - BERTopic - Cluster - Completed ✓
2026-11-18 23:13:27,348 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-11-18 23:21:04,667 - BERTopic - Representation - Completed ✓


C_V=0.7008 | UMass=-1.8348 | TD=0.8259 | IRBO=0.9951 | Topics=54 | Outliers=19.7%

Selected: {'n_neighbors': 50, 'min_cluster_size': 15, 'n_topics': 95, 'outlier_rate': 0.21706161137440758, 'coverage': 0.7829383886255924, 'c_v': 0.7247911684308647, 'u_mass': -2.043171757487749, 'topic_diversity': 0.8094736842105263, 'irbo': 0.9963043473034986, 'coherence_topics': 95, 'coherence_term_coverage': 0.5142105263157895}


,n_neighbors,min_cluster_size,n_topics,outlier_rate,coverage,c_v,u_mass,topic_diversity,irbo,coherence_topics,coherence_term_coverage
7,50,15,95,0.217062,0.782938,0.724791,-2.043172,0.809474,0.996304,95,0.514211
6,50,10,123,0.257109,0.742891,0.719875,-2.275780,0.809350,0.997090,123,0.488618
5,30,30,62,0.198934,0.801066,0.711876,-1.764160,0.808065,0.994643,62,0.557258
4,30,15,117,0.238033,0.761967,0.711506,-2.258861,0.798291,0.996860,117,0.493162
3,30,10,167,0.270853,0.729147,0.709638,-2.448489,0.768263,0.997337,167,0.458383
0,15,10,198,0.255095,0.744905,0.708765,-2.523697,0.756566,0.997382,198,0.454293
1,15,15,133,0.212559,0.787441,0.708703,-2.343304,0.786842,0.996807,133,0.485714
8,50,30,54,0.196801,0.803199,0.700750,-1.834792,0.825926,0.995097,54,0.556481
2,15,30,61,0.152844,0.847156,0.698445,-1.884240,0.819672,0.994371,61,0.557377


## 11. Save topics, model & experiment settings

In [15]:
assignments = df.reset_index(drop=True).copy()
assignments["topic"] = best_topics
topic_names = topic_model.get_topic_info().set_index("Topic")["Name"]
assignments["topic_name"] = assignments["topic"].map(topic_names)
assignments.to_csv(OUTPUT_DIR / "document_topics.csv", index=False, encoding="utf-8-sig")
topic_info = topic_model.get_topic_info()
topic_info.to_csv(OUTPUT_DIR / "topic_info.csv", index=False, encoding="utf-8-sig")

word_rows = [{"topic": k, "keywords": " | ".join(w for w, _ in topic_model.get_topic(k)[:TOP_WORDS])}
             for k in sorted(topic_model.get_topics()) if k != -1]
pd.DataFrame(word_rows).to_csv(OUTPUT_DIR / "topic_words.csv", index=False, encoding="utf-8-sig")

representatives = [{"topic": k, "rank": i, "text": t}
                   for k in sorted(topic_model.get_topics()) if k != -1
                   for i, t in enumerate(topic_model.get_representative_docs(k) or [], 1)]
pd.DataFrame(representatives, columns=["topic", "rank", "text"]).to_csv(
    OUTPUT_DIR / "representative_documents.csv", index=False, encoding="utf-8-sig")

experiment = {
    "field": TEXT_FIELD, "documents": len(docs), "seed": SEED,
    "embedding": cache_info, "embedding_batch_size": EMBEDDING_BATCH_SIZE,
    "umap": {**umap_model.get_params(), "n_neighbors": best_config["n_neighbors"]},
    "hdbscan": {**hdbscan_model.get_params(), "min_cluster_size": best_config["min_cluster_size"],
                "min_samples": hdbscan_model.min_samples},
    "vectorizer": {"ngram_range": vectorizer_model.ngram_range,
                   "min_df": vectorizer_model.min_df, "max_features": vectorizer_model.max_features,
                   "stopwords": sorted(TOPIC_STOPWORDS)},
    "ctfidf": {"reduce_frequent_words": ctfidf_model.reduce_frequent_words},
    "selection": "max c_v, min outlier_rate, min n_topics",
    "scores": best_config,
    "versions": {lib: version(lib) for lib in
                 ["bertopic", "sentence-transformers", "umap-learn", "hdbscan", "gensim", "underthesea"]},
}
(OUTPUT_DIR / "experiment_config.json").write_text(
    json.dumps(experiment, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

# topic_model.topic_sizes_ = {int(k): int(v) for k, v in topic_model.topic_sizes_.items()}
# topic_model.topic_representations_ = {
#     int(k): [(str(w), float(s)) for w, s in words]
#     for k, words in topic_model.topic_representations_.items()
# }

# topic_model.save(str(OUTPUT_DIR / "model"), serialization="safetensors",
#                  save_ctfidf=True, save_embedding_model=EMBEDDING_MODEL_ID)
print("Saved results:", OUTPUT_DIR)
display(topic_info.head(15))

Saved results: /kaggle/working/bertopic_fintechnews


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1832,-1_khách_hàng_công_nghệ_số_chuyển_đổi,"[khách_hàng, công_nghệ, số, chuyển_đổi, giúp, ...",[TIN MỚI Tốt nghiệp loại ưu ngành Kỹ thuật điệ...
1,0,661,0_bitcoin_giá bitcoin_giá_usd,"[bitcoin, giá bitcoin, giá, usd, btc, mức, tiề...",[TIN MỚI Giá Bitcoin hôm nay: Vượt 17.200 USD ...
2,1,507,1_thương_mại_điện_tử_hàng_bán hàng_tmđt,"[thương_mại_điện_tử, hàng, bán hàng, tmđt, bán...",[KTSG Online - Thương mại điện tử TMĐT đang hỗ...
3,2,397,2_tiền_mặt_thanh_toán_thanh_toán không_dùng ti...,"[tiền_mặt, thanh_toán, thanh_toán không, dùng ...",[TIN MỚI Đây là những thông tin đáng chú ý tại...
4,3,304,3_forex_sàn_nhà_đầu_tư_người tham_gia,"[forex, sàn, nhà_đầu_tư, người tham_gia, đa_cấ...",[TIN MỚI Ma trận kinh doanh mạng Kinh doanh đa...
5,4,262,4_vib_thẻ_thẻ_tín_dụng_chi_tiêu,"[vib, thẻ, thẻ_tín_dụng, chi_tiêu, chủ thẻ, ưu...","[TIN MỚI Theo đó, từ 15/3, VIB chính thức áp d..."
6,5,187,5_mb_msb_của mb_app_mbbank,"[mb, msb, của mb, app_mbbank, mbbank, tỷ đồng,...",[MBB: Giá hiện tại Thay đổi Xem hồ sơ doanh ng...
7,6,179,6_nhắn_tin nhắn_giả_mạo_link,"[nhắn, tin nhắn, giả_mạo, link, kẻ_gian, lừa_đ...",[TIN MỚI Giả mạo SMS Brand name - độc chiêu ha...
8,7,166,7_đổi_mới sáng_tạo_sáng_tạo_đổi_mới_khoa_học,"[đổi_mới sáng_tạo, sáng_tạo, đổi_mới, khoa_học...","[Bày tỏ những băn khoăn, trăn trở khi tốc độ t..."
9,8,126,8_xanh_tín_dụng xanh_thải_năng_lượng,"[xanh, tín_dụng xanh, thải, năng_lượng, bền_vữ...",[TIN MỚI Quan điểm trên được Phó Thống đốc Ngâ...


## 12. Visualizations — each in its own cell

Interactive Plotly figures are displayed inline and saved to HTML. No hard-coded topic IDs.

In [16]:
VIZ_DIR = OUTPUT_DIR / "visualizations"
VIZ_DIR.mkdir(exist_ok=True)

def save_figure(fig, name):
    location = VIZ_DIR / f"{name}.html"
    fig.write_html(str(location), include_plotlyjs=True)
    print("Saved:", location)
    if SHOW_PLOTS:
        display(fig)

n_topics = len([k for k in topic_model.get_topics() if k != -1])
print("Number of topics:", n_topics)

Number of topics: 95


### 12.1 Intertopic distance map

In [17]:
if n_topics >= 2:
    save_figure(topic_model.visualize_topics(), "01_intertopic_map")

Saved: /kaggle/working/bertopic_fintechnews/visualizations/01_intertopic_map.html


### 12.2 Topic hierarchy

In [18]:
if n_topics >= 2:
    save_figure(topic_model.visualize_hierarchy(), "02_hierarchy")

Saved: /kaggle/working/bertopic_fintechnews/visualizations/02_hierarchy.html


### 12.3 Topic keyword bar chart

In [19]:
if n_topics:
    save_figure(topic_model.visualize_barchart(top_n_topics=min(20, n_topics), n_words=10),
                "03_barchart")

Saved: /kaggle/working/bertopic_fintechnews/visualizations/03_barchart.html


### 12.4 Topic similarity heatmap

In [20]:
if n_topics >= 2:
    save_figure(topic_model.visualize_heatmap(top_n_topics=min(30, n_topics)), "04_heatmap")

Saved: /kaggle/working/bertopic_fintechnews/visualizations/04_heatmap.html


### 12.5 Document map

In [21]:
# Full data are used for training; a sample keeps the interactive plot responsive.
if n_topics >= 2:
    save_figure(topic_model.visualize_documents(
        docs, embeddings=embeddings, sample=0.20, hide_document_hover=True), "05_document_map")

Saved: /kaggle/working/bertopic_fintechnews/visualizations/05_document_map.html


### 12.6 Topics over time (optional date column)

In [22]:
if DATE_COLUMN not in df.columns:
    print(f"Skipped: date column {DATE_COLUMN!r} not found")
else:
    dates = pd.to_datetime(df[DATE_COLUMN], errors="coerce", utc=True)
    mask = dates.notna().to_numpy()
    if mask.sum() < 3 or dates[mask].nunique() < 2:
        print("Skipped: insufficient valid distinct dates")
    else:
        positions = np.flatnonzero(mask)
        topic_timeline = topic_model.topics_over_time(
            [docs[i] for i in positions],
            dates.iloc[positions].dt.tz_localize(None).tolist(),
            topics=best_topics[positions].tolist(),
            nr_bins=12,
        )
        topic_timeline.to_csv(OUTPUT_DIR / "topics_over_time.csv", index=False)
        save_figure(topic_model.visualize_topics_over_time(topic_timeline),
                    "06_topics_over_time")

12it [07:05, 35.48s/it]


Saved: /kaggle/working/bertopic_fintechnews/visualizations/06_topics_over_time.html


## 13. Inspect outputs

In [23]:
display(pd.read_csv(OUTPUT_DIR / "grid_search.csv"))
print("\nFiles:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file() and path.suffix in (".csv", ".json", ".html"):
        print(path)

,n_neighbors,min_cluster_size,n_topics,outlier_rate,coverage,c_v,u_mass,topic_diversity,irbo,coherence_topics,coherence_term_coverage
0,15,10,198,0.255095,0.744905,0.708765,-2.523697,0.756566,0.997382,198,0.454293
1,15,15,133,0.212559,0.787441,0.708703,-2.343304,0.786842,0.996807,133,0.485714
2,15,30,61,0.152844,0.847156,0.698445,-1.884240,0.819672,0.994371,61,0.557377
3,30,10,167,0.270853,0.729147,0.709638,-2.448489,0.768263,0.997337,167,0.458383
4,30,15,117,0.238033,0.761967,0.711506,-2.258861,0.798291,0.996860,117,0.493162
5,30,30,62,0.198934,0.801066,0.711876,-1.764160,0.808065,0.994643,62,0.557258
6,50,10,123,0.257109,0.742891,0.719875,-2.275780,0.809350,0.997090,123,0.488618
7,50,15,95,0.217062,0.782938,0.724791,-2.043172,0.809474,0.996304,95,0.514211
8,50,30,54,0.196801,0.803199,0.700750,-1.834792,0.825926,0.995097,54,0.556481



Files:
/kaggle/working/bertopic_fintechnews/document_topics.csv
/kaggle/working/bertopic_fintechnews/embeddings.json
/kaggle/working/bertopic_fintechnews/experiment_config.json
/kaggle/working/bertopic_fintechnews/grid_search.csv
/kaggle/working/bertopic_fintechnews/representative_documents.csv
/kaggle/working/bertopic_fintechnews/topic_info.csv
/kaggle/working/bertopic_fintechnews/topic_words.csv
/kaggle/working/bertopic_fintechnews/topics_over_time.csv
/kaggle/working/bertopic_fintechnews/visualizations/01_intertopic_map.html
/kaggle/working/bertopic_fintechnews/visualizations/02_hierarchy.html
/kaggle/working/bertopic_fintechnews/visualizations/03_barchart.html
/kaggle/working/bertopic_fintechnews/visualizations/04_heatmap.html
/kaggle/working/bertopic_fintechnews/visualizations/05_document_map.html
/kaggle/working/bertopic_fintechnews/visualizations/06_topics_over_time.html


In [24]:
import shutil
shutil.make_archive("/kaggle/working/bertopic_outputs", "zip", OUTPUT_DIR)

'/kaggle/working/bertopic_outputs.zip'

### Reproducibility notes

- A single `SEED=42` is used; UMAP is seeded explicitly. No topic stability/resampling evaluation.
- The embedding cache uses a SHA-256 digest of the exact documents and model settings.
- The saved manifest records model settings and dependency versions; results may still differ across hardware/software environments.
- To analyze another text column, change `TEXT_FIELD` and rerun all cells from data loading onward.
- `c_v` coherence assesses topic-word quality, not the overall correctness of document clusters.